<a href="https://colab.research.google.com/github/GustavoFA/IA368/blob/main/notebooks/8_prompt_eng.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# IA368-HH

## Prompt Engineering - Sentiment Analysis

Gustavo Freitas Alves

236249


---
### Imports

In [ ]:
!pip install -U -q datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.6/503.6 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 MB 17.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu12 25.6.0 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 21.0.0 which is incompatible.
pylibcudf-cu12 25.6.0 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 21.0.0 which is incompatible.


In [ ]:
import re
import random
from tqdm import tqdm
from html import unescape
from openai import OpenAI
from datasets import load_dataset

---
### Dados

Importação dos dados

In [ ]:
%%capture
imdb_dic = load_dataset("stanfordnlp/imdb", token=False)

IMPORTANTE: nesta atividade vou utilizar o dataset de treino para gerar os exemplos para o modelo (tanto para one-shot e few-shot quanto para o CoT). Já o dataset de teste servirá como os textos que serão analisados pelos modelos.

In [ ]:
len(imdb_dic['test']), len(imdb_dic['train'])

(25000, 25000)

Função para limpeza dos dados do dataset

In [ ]:
def clean_text(text:str) -> str:
  '''
    Limpeza geral dos textos
  '''
  # remover tags e entidades de HTML
  text = re.sub(r'<.*?>', '', text)
  text = unescape(text)

  # remover URLs e e-mails
  text = re.sub(r'http\S+|www\S+|@\S+', '', text)

  # remover múltiplos espaços e quebras de linha
  text = re.sub(r'\s+', ' ', text)

  # ajustar aspas dos textos
  try:
        text = text.encode('utf-8').decode('unicode_escape')
  except Exception:
      pass
  text = text.replace("\\'", "'").replace('\\"', '"')


  return text.strip()

Função para transformar os rótulos numéricos em palavras

In [ ]:
def label_to_str(label:int) -> str:
  '''
    Converte label para string
  '''
  return 'positive' if label == 1 else 'negative'

#### Dataset de reviews para utilizar como exemplos (treino) no *few-shot* e no CoT

In [ ]:
%%time
%%capture
all_examples = [(label_to_str(example['label']), clean_text(example['text'])) for example in imdb_dic['train']]

CPU times: user 5.75 s, sys: 36.8 ms, total: 5.78 s
Wall time: 9.2 s


Na lista completa, temos que a primeira metade são casos de reviews negativas e a outra metade são reviews positivas

In [ ]:
all_examples[random.randint(0, len(all_examples)-1)]

('negative',
 "The only redeeming quality of this overlong miscast melodrama is the scenery of southern France and the voice of Nana Mascouri singing the theme song. Stephanie Powers is miscast and betrayed by a phony accent. As has been pointed out, she is too old to play an 18 year old and looks far too young as a grandmother with a college age granddaughter? Lee Remick is good although she also is ageless in her later years. The talented Joanna Lumley is under utilized and also manages to look forever young when her middle aged son (Robert Urich) finally marries Grandma Stephanie Powers. Stacey Keach's ceaseless arrogance makes you wonder what these women saw in him. Don't know how any viewer could relate to his excessive portrayal? The most credible performance is given by Ian Richardson, who makes the rest of the cast look like rank amateurs. It strains credulity that the handsome male suitors in this epic would remain ever single while they patiently await the subject of their af

Para o CoT selecionei 3 reviews negativas e 3 positivas para serem utilizadas na geração de raciocínio. Serão agrupados em uma lista, seguindo a ordem de que os 3 primeiros são as reviews negativas e os 3 últimos as positivas.

In [ ]:
cot_examples = []
cot_reasoning = [
    "The review is intensely negative, a conclusion immediately established by the reviewer's opening statement, which labels the film the worst sequel on the face of the world of movies. This extreme condemnation is supported by repeated rejections of the film's logic, specifically the assertion that the plot doesn't make since (sense) and is the stupidest movie ever due to its meta-premise (a killer targeting people making a movie about his previous crimes). The reviewer directly and forcefully discourages viewership with the imperative Don't watch this and warns readers not to waste the one precious hour of the runtime. Further criticism is leveled at the film's poor execution of genre, as it doesn't combine the original makes of horror, action, and crime, and the text concludes with a strong, definitive warning to parents about the movie's negative impact on viewers.",
    "The review is profoundly negative, which is established immediately when the reviewer declares, This film is terrible. Although the reviewer offers minimal, conditional compliments—stating the film looks great and some of the performances are OK—these positive points are immediately dismissed by the overarching sentiment that The bad things are everything else about it. The review then launches into several major criticisms: the plot is severely slow-paced, boy does it take its time; the ensemble of characters is universally unlikable; the character relationships are confusing, leaving the reviewer extremely frustrated; and the relentlessly grim tone is considered a failure in the structure of the story. Furthermore, the reviewer details numerous silly lapses in logic, citing examples like a body drifting miles upstream and implausible police actions (ignoring the telephone in the age of rapid media coverage). The central narrative omission is highlighted as a failure: the film completely misses the essential debate the men would have had after finding the body, which is the entire basis of the mystery. The reviewer concludes by summarizing their viewing experience as being simultaneously bored and confused, rejecting a late thematic twist as ludicrous and incoherent, thereby firmly establishing the review as a comprehensive condemnation of the film's structure, logic, pacing, and overall merit.",
    "The review is distinctly negative, immediately established by the reviewer's inability to endure the content, stating I sat through almost one episode of this series and just couldn't take anymore. The core criticism is a fundamental lack of originality, summarized by the revelation There's nothing new here, followed by specific claims that its jokes and storylines are derivative, having been seen previously on shows like Seinfeld, Friends, and Happy Days. This suggests the series offers no fresh entertainment value. Furthermore, the reviewer condemns the cast, finding none of the actors are interesting here either, and harshly suggests that some actors are unsuited for the profession. The conclusion reinforces the intensely negative stance with a strong call to action: Avoid this stinker!, positioning the series as something to be completely shunned.",
    "The review is emphatically positive, immediately declaring the film Another good Stooge short. The praise is then directed toward specific actors, singling out Christine McIntyre for being so lovely and evil and the same time and calling her such a great actress, indicating appreciation for her performance quality and complexity. Further commendation is given to the main performers, noting The Stooges are very good and especially Shemp and Larry. Finally, the reviewer offers a recommendation for optimal viewing, stating This to is a good one to watch around Autumn time, which suggests the film has enduring appeal and situational appropriateness. Every substantive statement contributes to a high opinion of the film and its cast.",
    "The review is highly positive, despite acknowledging the film's low-quality elements. The main positive driver is the reviewer's embrace of the film's perceived flaws, stating that the plot is corny and cliche, but that the very corniness (citing the example of the evil green fog removing a girl's clothes) and the soundtrack are precisely what make the movie so hilarious and great. This signals an enjoyment rooted in irony and appreciation for cheesy, low-budget filmmaking. Further positive reinforcement comes from specific details, such as the enjoyment of Ozzy Osbourne playing a preacher. The reviewer's final assessment confirms the positive view by recommending it for having a few friends over for a good laugh, suggesting high rewatch value for social entertainment, and concluding with a definitive endorsement: A must see for cheesy movie fans.",
    "The review is ultimately positive, though it contains some reservations. The positive sentiment is established through direct praise for the performances, noting the two young actresses, Mischa Barton and Ingrid Uribe, played their roles nicely and intelligently, and that the acting is well done by all concerned. Although the reviewer critiques the plot as a stretch of the imagination and mentions the lack of a genuine atmosphere of drama, these issues are immediately mitigated by the philosophical observation that some films are simply good in their own way, like comparing Pollyanna to How Green Was My Valley. The review praises the casting, stating I do admire Joan Plowright and commends the film's suitability as middle of the road entertainment well suited for younger viewers. Strong positive endorsements are given to the inclusion of fine classical music (a rarity), and the overall thematic content, which is found to be a welcomed change as it reflects quieter, thoughtful values and contains no violence thank goodness. The concluding statement, A warm family film to enjoy, confirms the positive recommendation."
]
# selecionei algumas reviews aleatóriamente
for k, i in enumerate([201, 8954, 12001, 12591, 18432, 23999]):
  label, text= all_examples[i]
  label = label_to_str(label)
  cot_examples.append((label, text, cot_reasoning[k]))

In [ ]:
cot_examples

[('negative',
  "This is the worst sequel on the face of the world of movies. Once again it doesn't make since. The killer still kills for fun. But this time he is killing people that are making a movie about what happened in the first movie. Which means that it is the stupidest movie ever.Don't watch this. If you value the one precious hour during this movie then don't watch it. You'll want to ask the director and the person beside you what made him make it. Because it just doesn't combine the original makes of horror, action, and crime.Don't let your children watch this. Teenager, young child or young adult, this movie has that sorta impact upon people.",
  "The review is intensely negative, a conclusion immediately established by the reviewer's opening statement, which labels the film the worst sequel on the face of the world of movies. This extreme condemnation is supported by repeated rejections of the film's logic, specifically the assertion that the plot doesn't make since (sens

#### Dataset para verificação (teste) com o modelo deste projeto

In [ ]:
N_SAMPLES = 1000 # negativas + positivas

In [ ]:
%%time
%%capture
all_reviews = [
    (label_to_str(review['label']), clean_text(review['text']))
    for review in imdb_dic['test']
]

CPU times: user 4.14 s, sys: 56.5 ms, total: 4.2 s
Wall time: 4.21 s


In [ ]:
reviews = random.sample(all_reviews[:len(all_reviews)//2], N_SAMPLES//2) + random.sample(all_reviews[len(all_reviews)//2:], N_SAMPLES//2)

Verificando algumas reviews

In [ ]:
for j in random.sample(range(0, len(reviews)), 3):
  print(reviews[j])

('negative', 'Yes, it was an awful movie, but there was a song near the beginning of the movie, I think, called "I got a Woody" or something to that effect. I would love to find a sound track of that if there is one available. I saw this song on MST 3K, and as awful as it was, it had it\'s moments, and that song was one of them.If you like babes in bikinis, this is the movie for you, but if you don\'t, then don\'t bother. It was great material for MST 3K, I have to admit though. I would really love to know where to get a copy of the soundtrack though. Not just that song, but a couple more were really funny, and are classics as far as I\'m concerned.')
('positive', 'The Stepford Children, besides being a very good made for TV movie, shows the very disturbing result of indoctrination. It is quite a statement about how being made to act within the confines of what is considered "Good" behavior can destroy whatever it is that makes a person unique and an individual. I think that this is a 

---
### OpenAI

Chave

In [ ]:
# Insert here your key
KEY = ''

Modelo da OpenAI

In [ ]:
MODEL_OPENAI = 'gpt-5-nano'
REASONING = 'minimal'

Iniciando cliente

In [ ]:
client = OpenAI(api_key=KEY)

Geração

In [ ]:
def generate(prompt, model=MODEL_OPENAI, reasoning=REASONING):
  """
    Gerador de respostas utilizando modelo da OpenAI
  """
  response = client.responses.create(
      model=model,
      input=prompt,
      text={
        "format": {
            "type": "text"
          },
        "verbosity": "medium"
      },
      reasoning={
        "effort": reasoning
      }
  )

  return response.output_text

---
### Análise de sentimento pelo modelo gpt-5-nano

#### Zero-Shot

Apenas indico para o modelo que a resposta deve ser positiva e negativa

In [ ]:
def zero_shot_prompt(text:str) -> str:
  """
    Zero shot prompt
  """
  return f'''
  Analyze the sentiment of the following text and answer with only one word: Positive or Negative.
  Text: {text}
  '''

In [ ]:
def zero_shot_function():

  results = {
      'labels': [],
      'prompt': [],
      'results': [],
      'bad_answers': []
  }

  for review in tqdm(reviews,ncols=100):
    label, text = review
    prompt = zero_shot_prompt(text)
    result = generate(prompt)
    results['labels'].append(label)
    results['prompt'].append(prompt)
    results['results'].append(result)

  matches = [1 if label.lower() == result.lower() else 0
            for label, result in zip(results['labels'], results['results'])
            ]

  results['bad_answers'] = [i for i, _match in enumerate(matches) if _match == 0]

  print(f'\n\nNúmero de respostas corretas: {sum(matches)}/{len(matches)}')
  if len(matches) > 0:
    print(f'Acurácia: {sum(matches)/len(matches):.2%}')
  print(f'Número de respostas incorretas: {len(results["bad_answers"])} [{len(results["bad_answers"])/len(matches):.2%}]')
  if len(results['bad_answers']) > 0:
    print('\n\nAlgumas amostras com respostas incorretas:\n')
    for i in results['bad_answers'][:3]:
      print(f'Label: {results["labels"][i].lower()} | Result: {results["results"][i].lower()}')
      print(f'Prompt: {results["prompt"][i]}')
      print(30*'-')

  return results

In [ ]:
zero_shot_results = zero_shot_function()

100%|███████████████████████████████████████████████████████████| 1000/1000 [28:56<00:00,  1.74s/it]



Número de respostas corretas: 947/1000
Acurácia: 94.70%
Número de respostas incorretas: 53 [5.30%]


Algumas amostras com respostas incorretas:

Label: negative | Result: positive
Prompt: 
  Analyze the sentiment of the following text and answer with only one word: Positive or Negative.
  Text: This is a poorly written and badly directed short film, pure and simple. What is interesting and keep me watching, to some extent, was the production values. Shot on video it appears, with a bad script and bad direction, one would think it would also have horrible production value. That is what the viewer expects when they watch a film that is terrible and shot on video. BUT Not in this case, they spent some money and it shows. It keep me very mildly interested to see what was coming next, just to see!Probably the worst short film I have seen that looked big budget hollywood even though it was shot on some sort of video format.Instead of spending the sum of money they must have spent for some 

#### One-Shot

Forneço um texto, com a resposta, como exemplo da tarefa

In [ ]:
def n_shot_prompt(text:str, examples:list) -> str:
  """
    n shot prompt
  """

  examples_texts = ""
  for label, text_ex in examples:
    examples_texts += f'Text: {text_ex}\nAnswer: {label.lower()}\n\n'

  return f'''
  Considering the previous analysis examples:
  {examples_texts}
  Analyze the sentiment of the following text and answer with only one word: Positive or Negative.
  Text: {text}
  '''

In [ ]:
def one_shot_function():

  results = {
      'labels': [],
      'prompt': [],
      'example': [],
      'results': [],
      'bad_answers': []
  }

  for review in tqdm(reviews,ncols=100):
    label, text = review
    example = random.choice(all_examples)
    prompt = n_shot_prompt(text, [example])
    result = generate(prompt)
    results['labels'].append(label)
    results['prompt'].append(prompt)
    results['example'].append(example)
    results['results'].append(result)

  matches = [1 if label.lower() == result.lower() else 0
            for label, result in zip(results['labels'], results['results'])
            ]

  results['bad_answers'] = [i for i, _match in enumerate(matches) if _match == 0]

  print(f'\n\nNúmero de respostas corretas: {sum(matches)}/{len(matches)}')
  if len(matches) > 0:
    print(f'Acurácia: {sum(matches)/len(matches):.2%}')
  print(f'Número de respostas incorretas: {len(results["bad_answers"])} [{len(results["bad_answers"])/len(matches):.2%}]\n\n')
  if len(results['bad_answers']) > 0:
    print(35*"=")
    print('\nAlgumas amostras com respostas incorretas:\n')
    for i in results['bad_answers'][:3]:
      print(f'Label: {results["labels"][i].lower()} | Result: {results["results"][i].lower()}')
      print(f'Prompt: {results["prompt"][i]}')
      print(30*'-')

  return results

In [ ]:
one_shot_results = one_shot_function()

100%|███████████████████████████████████████████████████████████| 1000/1000 [29:12<00:00,  1.75s/it]



Número de respostas corretas: 942/1000
Acurácia: 94.20%
Número de respostas incorretas: 58 [5.80%]



Algumas amostras com respostas incorretas:

Label: negative | Result: positive
Prompt: 
  Considering the previous analysis examples:
  Text: Story says that on that on December 28, 1895, a small group of thirty-three people was gathered at Paris's Salon Indien Du Grand CafÃ© to witness the CinÃ©matographe, a supposedly new invention that resulted from the work done by a couple of photographers named August and Louis LumiÃ¨re. The small audience reunited that day (some by invitation, most due to curiosity) didn't really know what to expect from the show, and when a stationary photograph appeared projected on a screen, most thought that the CinÃ©matographe was just another fancy devise to present slide-show projections. Until the photograph started to move. What those thirty-three people experienced in awe that cold day of December was the very first public screening of a moving pictu

#### Few-Shot

Forneço alguns exemplos. Aqui faço testes com 2, 3 e 5 exemplos no prompt

In [ ]:
def few_shot_function(n_shots:int=2):

  results = {
      'labels': [],
      'prompt': [],
      'example': [],
      'results': [],
      'bad_answers': []
  }

  for review in tqdm(reviews,ncols=100):
    label, text = review
    example = random.sample(all_examples, n_shots)
    prompt = n_shot_prompt(text, example)
    result = generate(prompt)
    results['labels'].append(label)
    results['prompt'].append(prompt)
    results['example'].append(example)
    results['results'].append(result)

  matches = [1 if label.lower() == result.lower() else 0
            for label, result in zip(results['labels'], results['results'])
            ]

  results['bad_answers'] = [i for i, _match in enumerate(matches) if _match == 0]

  print(f'\n\nNúmero de respostas corretas: {sum(matches)}/{len(matches)}')
  if len(matches) > 0:
    print(f'Acurácia: {sum(matches)/len(matches):.2%}')
  print(f'Número de respostas incorretas: {len(results["bad_answers"])} [{len(results["bad_answers"])/len(matches):.2%}]\n\n')
  if len(results['bad_answers']) > 0:
    print(35*"=")
    print('\nAlgumas amostras com respostas incorretas:\n')
    for i in results['bad_answers'][:3]:
      print(f'Label: {results["labels"][i].lower()} | Result: {results["results"][i].lower()}')
      print(f'Prompt: {results["prompt"][i]}')
      print(30*'-')

  return results

2 exemplos

In [ ]:
_2_shot_results = few_shot_function()

100%|███████████████████████████████████████████████████████████| 1000/1000 [31:42<00:00,  1.90s/it]



Número de respostas corretas: 946/1000
Acurácia: 94.60%
Número de respostas incorretas: 54 [5.40%]



Algumas amostras com respostas incorretas:

Label: negative | Result: positive
Prompt: 
  Considering the previous analysis examples:
  Text: I love Kristen Dunst, especially in Elizabethtown. I guess she's the kind of actress who had better not act before camera, but just be herself. She did that, and she looked so natural in Elizabethtown. In this movie, however, she did try to add in more artificial performance, especially in the first half of the film, so that she looked more like a sober editor. While in the other half, she totally set herself back in her daily track, and I just couldn't tell her to be an editor any way. Therefore, her performance is not enduring in this film. The film,on the whole, is attracting and inspiring. the character of Young is full and reasonable. Anyway, the film tells a big and sophisticated story. The only big defect is that it didn't show a turning

3 exemplos

In [ ]:
_3_shot_results = few_shot_function(3)

100%|███████████████████████████████████████████████████████████| 1000/1000 [31:55<00:00,  1.92s/it]



Número de respostas corretas: 942/1000
Acurácia: 94.20%
Número de respostas incorretas: 58 [5.80%]



Algumas amostras com respostas incorretas:

Label: negative | Result: positive
Prompt: 
  Considering the previous analysis examples:
  Text: I didn't know much about this movie before I watched it, but I heard it had something to do with quantum physics so I was interested. What I didn't know is that this is NOT ACTUALLY A STORY but a bunch of New-Age blowhards who love the sound of their own voice talking about how little they know about basic quantum mechanics. I say it belongs more in the Documentary category than Comedy or Drama.Marlee Matlin is in the movie, in order to give this New Age symposium *some* sort of a storyline. Her portions of the film feel horribly tacked on and are meant to display the speaker's thoughts so we won't die of boredom. Matlin has a real job as a photographer, unlike the New Age hippie that crashes on her couch. We get to listen to nameless people ra

5 exemplos

In [ ]:
_5_shot_results = few_shot_function(5)

100%|███████████████████████████████████████████████████████████| 1000/1000 [30:45<00:00,  1.85s/it]



Número de respostas corretas: 950/1000
Acurácia: 95.00%
Número de respostas incorretas: 50 [5.00%]



Algumas amostras com respostas incorretas:

Label: negative | Result: positive
Prompt: 
  Considering the previous analysis examples:
  Text: I don't see enough TV game shows to understand the attraction of SHOW ME THE MONEY, but I suppose it holds some appeal for undemanding audiences. Ostensibly a quiz show, it offers contestants huge sums of money for answering a few simple questions. However, its quiz elements play only a small part in the proceedings, which I find tortuously complicated. For example, before answering a question, a contestant selects which question is to be asked by choosing from among random "A," "B," or "C" choices. Does this serve any purpose other than to slow the game down? It would be a lot quicker simply to start with "A." Contestants can pass on questions, but must answer one of the three questions in each category.After responding to a question, the cont

#### CoT (Chain-of-Thought)

Forneço um exemplo com resposta e raciocínio (no caso, utilizei linhas de pensamento do gemini)

In [ ]:
def cot_prompt(text_in:str, example:tuple, ) -> str:
  """
    Chain of thought prompt
  """
  label, text, reasoning = example
  return f"""

  Considering the previous review analysis:

  Text: {text}
  Reasoning: {reasoning}
  Sentiment: {label}

  Analyze the sentiment of the text below and explain your reasoning and, at the end, state whether it is positive or negative.
  Text:{text_in}
  Respond in the format:
  Reasoning: ...
  Sentiment: positive or negative
  """

In [ ]:
def cot_function():
  """
    Chain of thought function
  """

  results = {
      'labels': [],
      'prompt': [],
      'example': [],
      'results': [],
      'bad_answers': []
  }

  for review in tqdm(reviews,ncols=100):
    label, text = review
    example = random.choice(cot_examples)
    prompt = cot_prompt(text, example)
    result = generate(prompt)
    results['labels'].append(label)
    results['prompt'].append(prompt)
    results['example'].append(example)
    results['results'].append(result)

  matches = [1 if label.lower() == result.split("Sentiment:")[-1].strip().lower() else 0
            for label, result in zip(results['labels'], results['results'])
            ]

  results['bad_answers'] = [i for i, _match in enumerate(matches) if _match == 0]

  print(f'\n\nNúmero de respostas corretas: {sum(matches)}/{len(matches)}')
  if len(matches) > 0:
    print(f'Acurácia: {sum(matches)/len(matches):.2%}')
  print(f'Número de respostas incorretas: {len(results["bad_answers"])} [{len(results["bad_answers"])/len(matches):.2%}]\n\n')
  if len(results['bad_answers']) > 0:
    print(35*"=")
    print('\nAlgumas amostras com respostas incorretas:\n')
    for i in results['bad_answers'][:3]:
      print(f'Label: {results["labels"][i].lower()} | Result: {result.split("Sentiment:")[-1].strip().lower()}')
      print(f'Prompt: {results["prompt"][i]}')
      print(30*'-')

  return results

In [ ]:
cot_results = cot_function()

100%|███████████████████████████████████████████████████████████| 1000/1000 [36:04<00:00,  2.16s/it]



Número de respostas corretas: 941/1000
Acurácia: 94.10%
Número de respostas incorretas: 59 [5.90%]



Algumas amostras com respostas incorretas:

Label: negative | Result: positive
Prompt: 

  Considering the previous review analysis:

  Text: This film is terrible. I was really looking forward to it, as I thought "Lantana" was great.The following review may contain *spoilers******First, the good things: it looks great, some of the performances are OK. The bad things are everything else about it. The story, as you possibly know, is about some blokes who go fishing and discover a body, with the twist that they find it on Friday but continue fishing and finally report it on Sunday when they get back into mobile (cell phone) range. However the film takes it's time (boy does it take its time) getting to this central event.Of the ensemble of characters (about a dozen), not one seems to like another one (which is, I suppose, consistent, because they are all unlikable). I was extremely frus